In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, percentile_approx, trim, initcap, upper, round, when, concat_ws, year
from pyspark.sql.types import DateType, DoubleType, IntegerType, StringType

In [2]:
spark = (
    SparkSession.builder
    .appName("TradeCorpNettoyage")
    .getOrCreate()
)

In [3]:
base_path = "/home/jovyan/data/raw"

df_categories = spark.read.csv(
    f"{base_path}/categories.csv",
    header=True,
    inferSchema=True
)

df_customers = spark.read.csv(
    f"{base_path}/customers.csv",
    header=True,
    inferSchema=True
)

df_employees = spark.read.csv(
    f"{base_path}/employees.csv",
    header=True,
    inferSchema=True
)

df_order_details = spark.read.csv(
    f"{base_path}/order_details.csv",
    header=True,
    inferSchema=True
)

df_orders = spark.read.csv(
    f"{base_path}/orders.csv",
    header=True,
    inferSchema=True
)

df_products = spark.read.csv(
    f"{base_path}/products.csv",
    header=True,
    inferSchema=True
)

df_shippers = spark.read.csv(
    f"{base_path}/shippers.csv",
    header=True,
    inferSchema=True
)

df_suppliers = spark.read.csv(
    f"{base_path}/suppliers.csv",
    header=True,
    inferSchema=True
)

## Q11.

In [4]:

dataframes = {
    "categories": df_categories,
    "customers": df_customers,
    "employees": df_employees,
    "order_details": df_order_details,
    "orders": df_orders,
    "products": df_products,
    "shippers": df_shippers,
    "suppliers": df_suppliers
}

for name, df in dataframes.items():
    print(f"\n===== {name.upper()} =====")

    for c in df.columns:
        nb_nulls = df.filter(col(c).isNull()).count()
        print(f"{c} : {nb_nulls}")


===== CATEGORIES =====
category_id : 0
category_name : 0
description : 0
picture : 8

===== CUSTOMERS =====
customer_id : 0
company_name : 0
contact_name : 0
contact_title : 0
address : 0
city : 0
region : 60
postal_code : 1
country : 0
phone : 0
fax : 22

===== EMPLOYEES =====
employee_id : 0
last_name : 0
first_name : 0
title : 0
title_of_courtesy : 0
birth_date : 0
hire_date : 0
address : 0
city : 0
region : 4
postal_code : 0
country : 0
home_phone : 0
extension : 0
photo : 9
notes : 0
reports_to : 1
photo_path : 0

===== ORDER_DETAILS =====
order_id : 0
product_id : 0
unit_price : 0
quantity : 0
discount : 0

===== ORDERS =====
order_id : 0
customer_id : 0
employee_id : 0
order_date : 0
required_date : 0
shipped_date : 21
ship_via : 0
freight : 0
ship_name : 0
ship_address : 0
ship_city : 0
ship_region : 507
ship_postal_code : 19
ship_country : 0

===== PRODUCTS =====
product_id : 0
product_name : 0
supplier_id : 0
category_id : 0
quantity_per_unit : 0
unit_price : 0
units_in_stoc

## Q12.

In [5]:
# Supprimer les commandes non livrées
df_orders = df_orders.filter(col("shipped_date").isNotNull())

# Calculer la médiane de unit_price
median_unit_price = (
    df_products
    .select(percentile_approx("unit_price", 0.5).alias("median"))
    .collect()[0]["median"]
)

# Remplacer les valeurs nulles de unit_price par la médiane
df_products = df_products.fillna(
    {"unit_price": median_unit_price}
)

## Q13.

In [6]:
# df_orders : cast des colonnes de dates
df_orders = (
    df_orders
    .withColumn("order_date", col("order_date").cast(DateType()))
    .withColumn("required_date", col("required_date").cast(DateType()))
    .withColumn("shipped_date", col("shipped_date").cast(DateType()))
)
df_orders.show()

# df_order_details : cast des types numériques
df_order_details = (
    df_order_details
    .withColumn("unit_price", col("unit_price").cast(DoubleType()))
    .withColumn("quantity", col("quantity").cast(IntegerType()))
)
df_order_details.show()

+--------+-----------+-----------+----------+-------------+------------+--------+-------+--------------------+--------------------+--------------+-----------+----------------+------------+
|order_id|customer_id|employee_id|order_date|required_date|shipped_date|ship_via|freight|           ship_name|        ship_address|     ship_city|ship_region|ship_postal_code|ship_country|
+--------+-----------+-----------+----------+-------------+------------+--------+-------+--------------------+--------------------+--------------+-----------+----------------+------------+
|   10248|      VINET|          5|1996-07-04|   1996-08-01|  1996-07-16|       3|  32.38|Vins et alcools C...|  59 rue de l'Abbaye|         Reims|       NULL|           51100|      France|
|   10249|      TOMSP|          6|1996-07-05|   1996-08-16|  1996-07-10|       1|  11.61|  Toms Spezialitäten|       Luisenstr. 48|       Münster|       NULL|           44087|     Germany|
|   10250|      HANAR|          4|1996-07-08|   1996-08

## Q14.

In [7]:
#  TRIM les colonnes texte
for field in df_customers.schema.fields:
    if isinstance(field.dataType, StringType):
        df_customers = df_customers.withColumn(
            field.name,
            trim(col(field.name))
        )

# contact_name en Title Case
df_customers = df_customers.withColumn(
    "contact_name",
    initcap(col("contact_name"))
)

# country en majuscules
df_customers = df_customers.withColumn(
    "country",
    upper(col("country"))
)

## Q15.


In [8]:
df_order_details = (
    df_order_details
    .withColumnRenamed("unit_price", "prix_unitaire")
    .withColumnRenamed("quantity", "quantite")
)

df_orders = df_orders.withColumnRenamed(
    "ship_via",
    "shipper_id"
)

## Q16.

In [9]:
df_order_details = df_order_details.withColumn(
    "sous_total", round(col("prix_unitaire") * col("quantite") * (1 - col("discount")),2)
)

## Q17.

In [10]:
# df_products : True si le produit est en stock
df_products = df_products.withColumn(
    "en_stock",
    when(col("units_in_stock") > 0, True).otherwise(False)
)

# df_orders : True si la commande a été expédiée
df_orders = df_orders.withColumn(
    "is_shipped",
    when(col("shipped_date").isNotNull(), True).otherwise(False)
)


## Q18.

In [11]:
# Nombre total de lignes
total_customers = df_customers.count()

# Nombre de customer_id distincts
distinct_customers = df_customers.select("customer_id").distinct().count()

print("Nombre total de lignes :", total_customers)
print("Nombre de customer_id distincts :", distinct_customers)

# Suppression des doublons
if total_customers != distinct_customers:
    df_customers = df_customers.dropDuplicates(["customer_id"])
    print("Doublons supprimés.")
else:
    print("Aucun doublon sur customer_id.")

Nombre total de lignes : 91
Nombre de customer_id distincts : 91
Aucun doublon sur customer_id.


## Q19.

In [12]:
# # Garder uniquement les commandes de 1997
# df_orders_1997 = df_orders.filter(
#     year(col("order_date")) == 1997
# )

# # Garder uniquement les produits en stock et non discontinués
# df_products_disponibles = df_products.filter(
#     (col("units_in_stock") > 0) &
#     (col("discontinued") == 0)
# )

In [13]:
# Garder uniquement les commandes de 1997
df_orders = df_orders.filter(
    year(col("order_date")) == 1997
)

# Garder uniquement les produits en stock et non discontinués
df_products = df_products.filter(
    (col("units_in_stock") > 0) &
    (col("discontinued") == 0)
)

## Q20

In [14]:

df_employees = (
    df_employees
    .select(
        "employee_id",
        "first_name",
        "last_name",
        "title",
        "hire_date",
        "city",
        "country"
    )
    .withColumn(
        "full_name",
        concat_ws(" ", col("first_name"), col("last_name"))
    )
)

In [15]:
PATH = "/home/jovyan/data/tmp"

dataframes_clean = {
    "categories": df_categories,
    "customers": df_customers,
    "employees": df_employees,
    "order_details": df_order_details,
    "orders": df_orders,
    "products": df_products,
    "shippers": df_shippers,
    "suppliers": df_suppliers
}

for name, df in dataframes_clean.items():
    df.write \
        .mode("overwrite") \
        .parquet(f"{PATH}/{name}.parquet")

    print(f"{name}.parquet écrit avec succès")

categories.parquet écrit avec succès
customers.parquet écrit avec succès
employees.parquet écrit avec succès
order_details.parquet écrit avec succès
orders.parquet écrit avec succès
products.parquet écrit avec succès
shippers.parquet écrit avec succès
suppliers.parquet écrit avec succès
